# SQL Assessment: Banking & Financial Services

Welcome to the SQL assessment. This notebook uses an in-memory **DuckDB** database to evaluate your SQL querying skills on typical banking datasets.

**Instructions:**
1. Run the setup cell below to initialize the database and load the dummy data.
2. Review the Data Schema section to understand the tables.
3. Write your SQL queries inside the string variables provided for each question, then run the cell to see the output.

In [ ]:
!pip install duckdb
!pip install pandas

In [ ]:
import duckdb
import pandas as pd

# 1. Initialize an in-memory DuckDB connection
conn = duckdb.connect()

# 2. Create the Tables
conn.execute('''
CREATE TABLE transactions (txn_id VARCHAR, acc_id VARCHAR, txn_date TIMESTAMP, amount DECIMAL, txn_type VARCHAR, category VARCHAR);
CREATE TABLE lms_schedule (loan_id VARCHAR, inst_num INT, due_date DATE, inst_amount DECIMAL);
CREATE TABLE lms_payments (payment_id VARCHAR, loan_id VARCHAR, payment_date DATE, paid_amount DECIMAL);
CREATE TABLE bureau_reports (cust_id VARCHAR, bureau_name VARCHAR, score INT, report_date DATE);
CREATE TABLE customer_profiles (cust_id VARCHAR, acc_id VARCHAR, loan_id VARCHAR, monthly_income DECIMAL);
''')

# 3. Insert Dummy Data
conn.execute('''
INSERT INTO transactions VALUES 
('T1', 'A1', '2024-03-15 10:00:00', 6000, 'DR', 'Shopping'),
('T2', 'A1', '2024-03-20 12:00:00', 500, 'DR', 'Shopping'),
('T3', 'A1', '2024-03-21 11:00:00', 300, 'DR', 'Dining'),
('T4', 'A2', '2024-01-05 09:00:00', 2500, 'CR', 'Corporate'),
('T5', 'A2', '2024-02-05 09:00:00', 2500, 'CR', 'Corporate'),
('T6', 'A2', '2024-03-05 09:00:00', 2500, 'CR', 'Corporate'),
('T7', 'A3', '2024-03-10 08:00:00', 2500, 'DR', 'Transfer'),
('T8', 'A3', '2024-03-10 08:15:00', 2500, 'DR', 'Transfer'),
('T9', 'A3', '2024-03-10 08:30:00', 2500, 'DR', 'Transfer'),
('T10', 'A3', '2024-03-10 08:45:00', 2500, 'DR', 'Transfer'),
('T11', 'A3', '2024-03-10 08:50:00', 2500, 'DR', 'Transfer'),
('T12', 'A3', '2024-03-10 08:55:00', 2500, 'DR', 'Transfer');

INSERT INTO lms_schedule VALUES 
('L1', 1, '2023-12-01', 500), ('L1', 2, '2024-01-01', 500), ('L1', 3, '2024-02-01', 500),
('L2', 1, '2024-01-15', 1000), ('L2', 2, '2024-02-15', 1000);

INSERT INTO lms_payments VALUES 
('P1', 'L1', '2023-11-28', 500), ('P2', 'L1', '2023-12-30', 500), ('P3', 'L2', '2024-01-14', 1000);

INSERT INTO bureau_reports VALUES 
('C1', 'Experian', 720, '2024-03-01'), ('C1', 'Equifax', 735, '2024-03-01'), 
('C1', 'TransUnion', 710, '2024-02-15'), ('C2', 'Experian', 650, '2024-01-10');

INSERT INTO customer_profiles VALUES 
('C1', 'A1', 'L1', 8000), ('C2', 'A2', 'L2', 5000), ('C3', 'A3', NULL, 12000);
''')

# 4. Helper function to execute queries and return DataFrames
def run_query(query):
    try:
        return conn.execute(query).df()
    except Exception as e:
        return f"SQL Error: {str(e)}"

print("✅ Setup Complete: In-memory DuckDB initialized and dummy data loaded successfully!")

## Data Schema Overview

### 1. `transactions` (CASA Data)
Tracks all debits and credits for customer accounts.
| Column | Type | Description |
| --- | --- | --- |
| `txn_id` | VARCHAR | Unique transaction identifier |
| `acc_id` | VARCHAR | Account number |
| `txn_date` | TIMESTAMP | Date and time of transaction |
| `amount` | DECIMAL | Transaction value |
| `txn_type` | VARCHAR | 'CR' (Credit) or 'DR' (Debit) |
| `category` | VARCHAR | e.g., 'Corporate', 'ATM', 'Merchant', 'Transfer' |

### 2. `lms_schedule` (Repayment Plan)
The expected repayment timeline generated at loan disbursal.
| Column | Type | Description |
| --- | --- | --- |
| `loan_id` | VARCHAR | Unique loan identifier |
| `inst_num` | INT | Installment number (1, 2, 3...) |
| `due_date` | DATE | Date the payment is expected |
| `inst_amount` | DECIMAL | Total amount due (Principal + Interest) |

### 3. `lms_payments` (Actual Payments)
Actual payments received from the customer.
| Column | Type | Description |
| --- | --- | --- |
| `payment_id` | VARCHAR | Unique payment identifier |
| `loan_id` | VARCHAR | Loan identifier |
| `payment_date` | DATE | Date the payment was received |
| `paid_amount` | DECIMAL | Amount actually paid |

### 4. `bureau_reports` (Credit Bureau Data)
External data from providers like Experian or TransUnion.
| Column | Type | Description |
| --- | --- | --- |
| `cust_id` | VARCHAR | Customer identifier |
| `bureau_name` | VARCHAR | 'Experian', 'TransUnion', or 'Equifax' |
| `score` | INT | Credit score (e.g., 300-900) |
| `report_date` | DATE | Date the score was generated |

### 5. `customer_profiles` (Customer Meta-Data)
Core demographic and linkage file.
| Column | Type | Description |
| --- | --- | --- |
| `cust_id` | VARCHAR | Customer identifier |
| `acc_id` | VARCHAR | Account identifier |
| `loan_id` | VARCHAR | Loan identifier |
| `monthly_income` | DECIMAL | Customer's verified monthly income |

---
## Part 1: Warm-up

### Q1: High Value Transactions
Find all `acc_id`s that have made at least one 'Debit' transaction greater than $5,000 in the last 30 days.

In [ ]:
q1_sql = """
-- Write your query here
SELECT * FROM transactions LIMIT 5;
"""

run_query(q1_sql)

### Q2: Top Spenders by Category
For each account, find the `category` they spent the most money on (Debit) in the last calendar month.

In [ ]:
q2_sql = """
-- Write your query here

"""

run_query(q2_sql)

---
## Part 2: Transactional Analysis

### Q3: Identifying "Salary" Customers
A customer is considered a "Salary" customer if they receive at least one credit transaction greater than $2,000 from a 'Corporate' source for 3 consecutive months. Write a query to flag all customers who met this condition in the last 3 months.

In [ ]:
q3_sql = """
-- Write your query here

"""

run_query(q3_sql)

### Q4: Velocity Alerts for Fraud Detection
Write a query to identify accounts that have had more than 5 debit transactions within any 1-hour rolling window, where the total value of those transactions exceeded $10,000. Return the `account_id` and the `start_time` of the first transaction in that window.

In [ ]:
q4_sql = """
-- Write your query here

"""

run_query(q4_sql)

---
## Part 3: Loan Management System (LMS)

### Q5: NPA (Non-Performing Asset) Classification
Calculate the Days Past Due (DPD) based on the oldest unpaid installment. Assign the asset class for every active loan as of today using the following logic:
- **Standard:** 0 DPD
- **SMA-0:** 1-30 DPD
- **SMA-1:** 31-60 DPD
- **SMA-2:** 61-90 DPD
- **NPA:** > 90 DPD

In [ ]:
q5_sql = """
-- Write your query here

"""

run_query(q5_sql)

### Q6: Month-on-Month (MoM) Roll-Back Analysis
A "Roll-Back" occurs when a customer moves from a worse DPD bucket (e.g., SMA-1) in the previous month to a better bucket (e.g., SMA-0) in the current month. Write a query to calculate the percentage of customers in the SMA-1 bucket last month who successfully rolled back to SMA-0 this month.

In [ ]:
q6_sql = """
-- Write your query here

"""

run_query(q6_sql)

---
## Part 4: Credit Bureau & Analytics

### Q7: Multi-Bureau "Golden Score" Logic
For each customer, determine their "Active Golden Score". This score should be taken from the bureau that was updated most recently. If multiple bureaus were updated on the exact same day, take the highest score among them.

In [ ]:
q7_sql = """
-- Write your query here

"""

run_query(q7_sql)

### Q8: Debt-to-Income (DTI) Calculation
Calculate the Debt-to-Income (DTI) ratio for all customers. 

The DTI calculation logic is: `(Sum of Internal EMIs + Sum of Bureau Scheduled Payments) / Monthly Income`. 
Write a query to calculate this by joining internal LMS data with external Bureau files. *(Assume all scheduled `inst_amount` in `lms_schedule` represent internal EMI).* 

In [ ]:
q8_sql = """
-- Write your query here

"""

run_query(q8_sql)